# IPL Data Cleaning

This notebook is the first step in the IPL SQL analysis project. It loads the raw Kaggle CSVs (`matches.csv`, `deliveries.csv`), inspects them for data-quality issues, fixes what can be fixed in Python, and exports `matches_cleaned.csv` / `deliveries_cleaned.csv` for use in the SQL scripts that follow (`01_setup_and_load.sql` onward).

**Dataset:** [IPL Complete Dataset (2008-2024) - Kaggle](https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020)


## 1. Load the raw data

In [1]:
import pandas as pd

In [2]:
matches_df = pd.read_csv('matches.csv')
deliveries_df = pd.read_csv('deliveries.csv')
matches_df.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [3]:
deliveries_df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


## 2. Structure Overview

In [4]:
matches_df.shape

(1095, 20)

In [5]:
deliveries_df.shape

(260920, 17)

In [6]:
matches_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

In [7]:
deliveries_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          260920 non-null  int64 
 1   inning            260920 non-null  int64 
 2   batting_team      260920 non-null  object
 3   bowling_team      260920 non-null  object
 4   over              260920 non-null  int64 
 5   ball              260920 non-null  int64 
 6   batter            260920 non-null  object
 7   bowler            260920 non-null  object
 8   non_striker       260920 non-null  object
 9   batsman_runs      260920 non-null  int64 
 10  extra_runs        260920 non-null  int64 
 11  total_runs        260920 non-null  int64 
 12  extras_type       14125 non-null   object
 13  is_wicket         260920 non-null  int64 
 14  player_dismissed  12950 non-null   object
 15  dismissal_kind    12950 non-null   object
 16  fielder           9354 non-null    obj

In [8]:
matches_df.nunique()

id                 1095
season               17
city                 36
date                823
match_type            8
player_of_match     291
venue                58
team1                19
team2                19
toss_winner          19
toss_decision         2
winner               19
result                4
result_margin        98
target_runs         170
target_overs         15
super_over            2
method                1
umpire1              62
umpire2              62
dtype: int64

## 3. Missing Values

In [9]:
matches_df.isnull().sum()

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

In [10]:
deliveries_df.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

## 4. Duplicate & Referential Integrity Checks

Confirms there are no duplicate match IDs, no duplicate deliveries (at the match/inning/over/ball level), and that every match in `matches_df` has corresponding deliveries and vice versa.

In [11]:
matches_df['id'].duplicated().sum()

np.int64(0)

In [12]:
deliveries_df.duplicated().sum()

np.int64(0)

In [13]:
deliveries_df.duplicated(
    subset=['match_id', 'inning', 'over', 'ball']
).sum()

np.int64(0)

In [14]:
deliveries_df['match_id'].isin(matches_df['id']).all()

np.True_

In [15]:
matches_df['id'].isin(deliveries_df['match_id']).value_counts()

id
True    1095
Name: count, dtype: int64

## 5. Team Name Inconsistencies

In [16]:
sorted(matches_df['team1'].unique())

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Delhi Daredevils',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kings XI Punjab',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiant',
 'Rising Pune Supergiants',
 'Royal Challengers Bangalore',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [17]:
sorted(matches_df['team2'].unique())

['Chennai Super Kings',
 'Deccan Chargers',
 'Delhi Capitals',
 'Delhi Daredevils',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kings XI Punjab',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiant',
 'Rising Pune Supergiants',
 'Royal Challengers Bangalore',
 'Royal Challengers Bengaluru',
 'Sunrisers Hyderabad']

In [18]:
matches_df['team1'].value_counts()

team1
Royal Challengers Bangalore    135
Chennai Super Kings            128
Mumbai Indians                 123
Kolkata Knight Riders          121
Rajasthan Royals               101
Kings XI Punjab                 92
Sunrisers Hyderabad             86
Delhi Daredevils                85
Delhi Capitals                  41
Deccan Chargers                 39
Punjab Kings                    31
Lucknow Super Giants            23
Pune Warriors                   23
Gujarat Titans                  21
Gujarat Lions                   16
Royal Challengers Bengaluru      9
Kochi Tuskers Kerala             7
Rising Pune Supergiant           7
Rising Pune Supergiants          7
Name: count, dtype: int64

In [19]:
matches_df['team2'].value_counts()

team2
Mumbai Indians                 138
Kolkata Knight Riders          130
Rajasthan Royals               120
Chennai Super Kings            110
Royal Challengers Bangalore    105
Kings XI Punjab                 98
Sunrisers Hyderabad             96
Delhi Daredevils                76
Delhi Capitals                  50
Deccan Chargers                 36
Punjab Kings                    25
Gujarat Titans                  24
Pune Warriors                   23
Lucknow Super Giants            21
Gujarat Lions                   14
Rising Pune Supergiant           9
Kochi Tuskers Kerala             7
Rising Pune Supergiants          7
Royal Challengers Bengaluru      6
Name: count, dtype: int64

Several teams appear under more than one name across seasons:

- Delhi Daredevils, Delhi Capitals
- Kings XI Punjab, Punjab Kings
- Royal Challengers Bangalore, Royal Challengers Bengaluru
- Rising Pune Supergiant, Rising Pune Supergiants

These are genuine rebrands (same ownership, new name), not new franchises, so they should be merged for analysis:

| Raw name                    | Canonical name               |
| ---------------------------- | ---------------------------- |
| Delhi Daredevils            | Delhi Capitals               |
| Delhi Capitals              | Delhi Capitals                |
| Kings XI Punjab             | Punjab Kings                  |
| Punjab Kings                | Punjab Kings                  |
| Royal Challengers Bangalore | Royal Challengers Bengaluru   |
| Royal Challengers Bengaluru | Royal Challengers Bengaluru   |
| Rising Pune Supergiant      | Rising Pune Supergiants       |
| Rising Pune Supergiants     | Rising Pune Supergiants       |

**Note:** this mapping is not applied here in Python. It's implemented as a `team_name_mapping` lookup table in `02_team_name_cleaning.sql`, so the raw team names are preserved in the exported CSV and the mapping logic lives alongside the rest of the SQL analysis. A separate typo (`Pune Warriros` vs `Pune Warriors`) also turned up later, only in the `winner` column. See the SQL script for the full 20-row mapping.

Also note: Deccan Chargers -> Sunrisers Hyderabad and Pune Warriors -> Rising Pune Supergiant(s) are **not** merged. Those are different franchises that happened to play in the same city, not renames of the same team.

## 6. Season Format

Season values are inconsistently formatted (some as `'2007/08'`, others as `'2009'`). Left as-is here. It doesn't affect grouping in the SQL analysis since each string is still a unique, correct season label, but worth noting as a known quirk of the raw data.

In [20]:
matches_df['season'].unique()

array(['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022',
       '2023', '2024'], dtype=object)

## 7. Match Outcome & Metadata Distributions

In [21]:
matches_df['result'].value_counts(dropna=False)

result
wickets      578
runs         498
tie           14
no result      5
Name: count, dtype: int64

In [22]:
matches_df['match_type'].value_counts()

match_type
League                1029
Final                   17
Qualifier 2             14
Qualifier 1             14
Eliminator              11
Semi Final               6
Elimination Final        3
3rd Place Play-Off       1
Name: count, dtype: int64

In [23]:
matches_df['toss_decision'].value_counts()

toss_decision
field    704
bat      391
Name: count, dtype: int64

In [24]:
matches_df['super_over'].value_counts()

super_over
N    1081
Y      14
Name: count, dtype: int64

## 8. Missing Winners (No Result Matches)

5 matches have no winner. All genuine no-result games (rain-abandoned, no result possible).

In [25]:
matches_df['winner'].isna().sum()

np.int64(5)

In [26]:
matches_df[matches_df['winner'].isnull()]

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
241,501265,2011,Delhi,2011-05-21,League,NaN,Feroz Shah Kotla,Delhi Daredevils,Pune Warriors,Delhi Daredevils,bat,NaN,no result,NaN,NaN,NaN,N,NaN,SS Hazare,RJ Tucker
485,829763,2015,Bangalore,2015-04-29,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,NaN,NaN,N,NaN,JD Cloete,PG Pathak
511,829813,2015,Bangalore,2015-05-17,League,NaN,M Chinnaswamy Stadium,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,field,NaN,no result,NaN,188.0,20.0,N,NaN,HDPK Dharmasena,K Srinivasan
744,1178424,2019,Bengaluru,2019-04-30,League,NaN,M.Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals,Rajasthan Royals,field,NaN,no result,NaN,63.0,5.0,N,NaN,NJ Llong,UV Gandhe
994,1359519,2023,Lucknow,2023-05-03,League,NaN,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings,Chennai Super Kings,field,NaN,no result,NaN,NaN,NaN,N,NaN,AK Chaudhary,NA Patwardhan


**Downstream note:** `pandas.to_csv()` writes these `NaN` winners as empty strings (`''`) in the exported CSV, not as a true missing value. MySQL's `LOAD DATA INFILE` then loads them as empty strings rather than `NULL`, which silently breaks any `WHERE winner IS NOT NULL` filter. `02_team_name_cleaning.sql` explicitly re-normalizes these back to real `NULL` values after loading.

## 9. Multi-Innings Matches (Super Overs)

`inning` values above 2 correspond to Super Over deliveries.

In [27]:
deliveries_df['inning'].value_counts()

inning
1    135018
2    125741
3        77
4        72
5         8
6         4
Name: count, dtype: int64

In [28]:
deliveries_df[deliveries_df['inning'] > 2][
    ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball']
].head(20)

,match_id,inning,batting_team,bowling_team,over,ball
15417,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,1
15418,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,2
15419,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,3
15420,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,4
15421,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,5
15422,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,6
15423,392190,3,Kolkata Knight Riders,Rajasthan Royals,0,7
15424,392190,4,Rajasthan Royals,Kolkata Knight Riders,0,1
15425,392190,4,Rajasthan Royals,Kolkata Knight Riders,0,2
15426,392190,4,Rajasthan Royals,Kolkata Knight Riders,0,3


In [29]:
deliveries_df['inning'].unique()

array([1, 2, 3, 4, 5, 6])

## 10. Dismissal & Extras Types

These distributions inform the bowler-wicket and strike-rate logic used later in `03_analysis_queries.sql` (Q6 and Q9) — e.g. run outs are excluded from a bowler's wicket count, and wides are excluded from a batter's balls faced.

In [30]:
deliveries_df['dismissal_kind'].value_counts(dropna=False)

dismissal_kind
NaN                      247970
caught                     8063
bowled                     2212
run out                    1114
lbw                         800
caught and bowled           367
stumped                     358
retired hurt                 15
hit wicket                   15
obstructing the field         3
retired out                   3
Name: count, dtype: int64

In [31]:
deliveries_df['extras_type'].value_counts(dropna=False)

extras_type
NaN        246795
wides        8380
legbyes      4001
noballs      1069
byes          673
penalty         2
Name: count, dtype: int64

## 11. Fixing Missing City Values

51 matches have a missing `city`, all played at just two venues (Sharjah and Dubai) that were never tagged with a city in the raw data. Fixed by backfilling from the venue name.

In [32]:
matches_df[matches_df['city'].isnull()][['venue']].drop_duplicates()

,venue
399,Sharjah Cricket Stadium
402,Dubai International Cricket Stadium


In [33]:
matches_df.loc[matches_df['venue'] == 'Sharjah Cricket Stadium', 'city'] = 'Sharjah'
matches_df.loc[matches_df['venue'] == 'Dubai International Cricket Stadium', 'city'] = 'Dubai'

matches_df['city'].isnull().sum()  # confirm it's now 0

np.int64(0)

## 12. Date Range

In [34]:
matches_df['date'].min(), matches_df['date'].max()

('2008-04-18', '2024-05-26')

## 13. Export Cleaned Data

No rows were dropped and no team names were merged in this notebook. Only the city fix above was applied. Team name normalization, the empty-string-to-NULL fix, and all further cleaning happen in SQL, so that logic lives in one place alongside the rest of the analysis.

**Note on column names:** the `deliveries.csv` column is named `over`, while the SQL table calls it `over_num`. This is fine. `LOAD DATA INFILE` in `01_setup_and_load.sql` maps columns by position, not by header name, so the column name in the CSV doesn't need to match the SQL table's column name.

In [35]:
matches_df.to_csv('matches_cleaned.csv', index=False)
deliveries_df.to_csv('deliveries_cleaned.csv', index=False)

## Next Step

Continue with `01_setup_and_load.sql` to create the database tables and load these cleaned CSVs.